# British Sign Language Gesture Recognition with CNNs and MobileNetV2

This notebook compares three image-classification approaches for **11 static British Sign Language (BSL) alphabet gestures**:

1. A baseline convolutional neural network trained from scratch
2. MobileNetV2 used as a frozen feature extractor
3. Fine-tuned MobileNetV2

The coursework report records validation performance of roughly **75%**, **89%**, and **94%** respectively, with fine-tuned MobileNetV2 selected as the strongest model.

> The dataset is not distributed with this repository. Update the paths below to point to your local/Google Drive copy.

## 1. Imports and configuration

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Activation, Conv2D, Dense, Dropout, Flatten, InputLayer, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_ROWS = 224
IMG_COLS = 224
IMG_CHANNELS = 3
BATCH_SIZE = 32
BASELINE_EPOCHS = 20
TRANSFER_EPOCHS = 8
FINE_TUNE_EPOCHS = 8
VALIDATION_SPLIT = 0.20

# Original Colab locations used for the coursework. Change these as needed.
ZIP_PATH = Path('/content/drive/MyDrive/DNN/2_hand.zip')
EXTRACT_PATH = Path('/content/2_hand')
DATA_DIR = EXTRACT_PATH / '2_hand'

print('TensorFlow:', tf.__version__)
print('Dataset directory:', DATA_DIR)

## 2. Dataset preparation

The original dataset contained **11,000 images across 11 balanced classes**. The training notebook uses an 80/20 training-validation split.

In [ ]:
if ZIP_PATH.exists() and not DATA_DIR.exists():
    EXTRACT_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        archive.extractall(EXTRACT_PATH)
    print('Extracted dataset to:', EXTRACT_PATH)
elif DATA_DIR.exists():
    print('Dataset already available:', DATA_DIR)
else:
    print('Dataset not found. Update ZIP_PATH or DATA_DIR before continuing.')

In [ ]:
if DATA_DIR.exists():
    class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    counts = {p.name: sum(1 for f in p.iterdir() if f.suffix.lower() in image_exts) for p in class_dirs}
    print(f'Classes: {len(class_dirs)}')
    print(f'Total images: {sum(counts.values())}')
    print('Per-class counts:', counts)

## 3. Baseline CNN

In [ ]:
baseline_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=VALIDATION_SPLIT,
)

train_gen = baseline_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_ROWS, IMG_COLS),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)

val_gen = baseline_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_ROWS, IMG_COLS),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

NB_CLASSES = train_gen.num_classes
print('Number of classes:', NB_CLASSES)

In [ ]:
baseline_model = Sequential([
    InputLayer(shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS)),
    Conv2D(32, (3, 3), padding='same'),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    Flatten(),
    Dense(512),
    Activation('relu'),
    Dropout(0.5),
    Dense(NB_CLASSES),
    Activation('softmax'),
])

baseline_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(),
    metrics=['accuracy'],
)

baseline_model.summary()

In [ ]:
history_baseline = baseline_model.fit(
    train_gen,
    epochs=BASELINE_EPOCHS,
    validation_data=val_gen,
    verbose=1,
)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_baseline, 'Baseline CNN')

## 4. Transfer learning with frozen MobileNetV2

In [ ]:
transfer_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT,
)

train_gen_mnv2 = transfer_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_ROWS, IMG_COLS),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)

val_gen_mnv2 = transfer_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_ROWS, IMG_COLS),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

NB_CLASSES = train_gen_mnv2.num_classes

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS),
)
base_model.trainable = False

transfer_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NB_CLASSES, activation='softmax'),
])

transfer_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

transfer_model.summary()

In [ ]:
history_frozen = transfer_model.fit(
    train_gen_mnv2,
    epochs=TRANSFER_EPOCHS,
    validation_data=val_gen_mnv2,
    verbose=1,
)

plot_history(history_frozen, 'Frozen MobileNetV2')

## 5. Fine-tuning MobileNetV2

In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(int(layer.trainable) for layer in base_model.layers)
print('Trainable MobileNetV2 layers:', trainable_count)

transfer_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
history_finetuned = transfer_model.fit(
    train_gen_mnv2,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=val_gen_mnv2,
    verbose=1,
)

plot_history(history_finetuned, 'Fine-tuned MobileNetV2')

## 6. Model comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_baseline.history['val_accuracy'], label='Baseline CNN')
plt.plot(history_frozen.history['val_accuracy'], label='MobileNetV2 (Frozen)')
plt.plot(history_finetuned.history['val_accuracy'], label='MobileNetV2 (Fine-tuned)')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Validation confusion matrix

The original supplied notebook ended with a confusion-matrix cell that referenced undefined variables. The cleaned version below computes the matrix directly from the validation generator, so the notebook is self-contained and executable.

In [ ]:
val_gen_mnv2.reset()
probabilities = transfer_model.predict(val_gen_mnv2, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = val_gen_mnv2.classes
class_names = list(val_gen_mnv2.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    xticks_rotation=45,
    cmap='Blues',
    values_format='d',
)
plt.title('Fine-tuned MobileNetV2 - Validation Confusion Matrix')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 8. Conclusion

The project demonstrates the benefit of transfer learning for static BSL gesture recognition. According to the submitted report, the baseline CNN showed clear overfitting, frozen MobileNetV2 improved generalisation, and fine-tuning achieved the strongest validation performance at approximately **94%**.

For deployment-oriented future work, the next steps would include a fully reproducible held-out test split, additional recording conditions, more alphabet classes, and temporal modelling for dynamic gestures.